In [1]:
import pandas as pd

musk_tweets = pd.read_csv('../data/raw/all_musk_posts.csv', low_memory=False)
musk_quotes = pd.read_csv("../data/raw/musk_quote_tweets.csv", low_memory=False)

In [2]:
# --- Cleaning and Type Conversion ---

In [3]:
CLEANED_FILE_PATH = '../data/cleaned/musk_tweets_cleaned.csv'

#The 'createdAt' column must be a datetime object for time series analysis
musk_tweets['createdAt'] = pd.to_datetime(musk_tweets['createdAt'], utc=True, errors='raise')

# These columns are numerical but contain missing values
count_columns = [
    'retweetCount', 'replyCount', 'likeCount', 'quoteCount',
    'viewCount', 'bookmarkCount'
]

for col in count_columns:
    # Coerce any non-numeric values (like empty strings or N/A) to NaN (Not a Number), Then convert the entire column to float64
    musk_tweets[col] = pd.to_numeric(musk_tweets[col], errors='coerce').astype('float64')

# These columns contain mixed types (True/False strings, or NaN)
boolean_columns = [
    'isReply', 'isPinned', 'isRetweet', 'isQuote',
    'isConversationControlled', 'possiblySensitive'
]

for col in boolean_columns:
    # Standardize string representations: 'True' -> True, 'False' -> False
    # Then use .astype('bool') to convert to boolean. NaNs will remain as NaNs (if applicable).
    # NOTE: The .astype('bool') can be tricky with NaNs; using map is safer for conversion.
    musk_tweets[col] = musk_tweets[col].astype(str).map({'True': True, 'False': False, 'nan': False, 'None': False}).fillna(False).astype(bool)
    # Using fillna(False) treats all missing values in these binary flags as not having the property
    # If the tweet doesn't have a value for 'isRetweet', we assume it's not a retweet.

# 5. Convert Large ID Columns to Appropriate Integer Types
# ID columns are often better as int64. We can fill NaNs with 0 temporarily for integer conversion.
id_columns = ['id', 'inReplyToId', 'conversationId', 'inReplyToUserId', 'quoteId']
for col in id_columns:
    # Use downcast='integer' to save memory, and errors='coerce' to turn bad values to NaN.
    # We must fillna before converting to integer type to avoid errors.
    musk_tweets[col] = pd.to_numeric(musk_tweets[col], errors='coerce').fillna(0).astype('int64')

# 6. Optional: Convert High-Cardinality Text Columns to Categorical for Memory Saving
musk_tweets['inReplyToUsername'] = musk_tweets['inReplyToUsername'].astype('category')

# --- Save the Cleaned File ---

# Save the processed DataFrame to the 'data/cleaned' folder
musk_tweets.to_csv(CLEANED_FILE_PATH, index=False)

print(f"✅ Data cleaning complete. Cleaned file saved to: {CLEANED_FILE_PATH}")
print("\n--- Cleaned DataFrame Info ---")
musk_tweets.info()

✅ Data cleaning complete. Cleaned file saved to: ../data/cleaned/musk_tweets_cleaned.csv

--- Cleaned DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55099 entries, 0 to 55098
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   id                        55099 non-null  int64              
 1   url                       55099 non-null  object             
 2   twitterUrl                55099 non-null  object             
 3   fullText                  55099 non-null  object             
 4   retweetCount              55009 non-null  float64            
 5   replyCount                54297 non-null  float64            
 6   likeCount                 55009 non-null  float64            
 7   quoteCount                54271 non-null  float64            
 8   viewCount                 34455 non-null  float64            
 9   createdAt                 55

CLEANED_FILE_PATH = '../data/cleaned/musk_quotes_cleaned.csv'

# 2. Convert Critical Time Columns to datetime
datetime_columns = ['orig_tweet_created_at', 'musk_quote_created_at']
for col in datetime_columns:
    musk_quotes[col] = pd.to_datetime(musk_quotes[col], utc=True, errors='coerce')

# 3. Clean and Convert Engagement Count Columns to int64
count_columns = [
    'orig_tweet_retweet_count', 'orig_tweet_reply_count', 'orig_tweet_like_count',
    'orig_tweet_quote_count', 'orig_tweet_view_count', 'orig_tweet_bookmark_count',
    'musk_quote_retweet_count', 'musk_quote_reply_count', 'musk_quote_like_count',
    'musk_quote_quote_count', 'musk_quote_view_count', 'musk_quote_bookmark_count'
]
for col in count_columns:
    musk_quotes[col] = pd.to_numeric(musk_quotes[col], errors='coerce').fillna(0).astype('int64')

# 4. Convert ID Columns to Integer (int64)
id_columns = ['orig_tweet_id', 'musk_tweet_id']
for col in id_columns:
    musk_quotes[col] = pd.to_numeric(musk_quotes[col], errors='coerce').fillna(0).astype('int64')

# 5. Convert Categorical/String ID Columns to Category
musk_quotes['orig_tweet_username'] = musk_quotes['orig_tweet_username'].astype('category')

# Save the processed DataFrame
musk_quotes.to_csv(CLEANED_FILE_PATH, index=False)